In [ ]:
# ============================================================================
# 1. 설정 (Configuration)
# ============================================================================
import os
from pathlib import Path
from dotenv import load_dotenv

# .env 파일 로드
env_path = Path(__file__).parent / '.env' if '__file__' in globals() else Path.cwd() / '.env'
load_dotenv(env_path)

# 환경 변수에서 API 키 로드
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Pinecone 설정
PINECONE_CLOUD = "aws"
PINECONE_REGION = "us-east-1"
INDEX_NAME = "acquiror" 

XLSX_PATH = "data/output/acqurior_filtered_part2.xlsx"  # 상대 경로


In [13]:
# pinecone-client 제거 후 pinecone 설치
%pip uninstall pinecone-client -y
%pip install pinecone openai pandas openpyxl tqdm

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ============================================================================
# 2. 라이브러리 설치 및 임포트
# ============================================================================

# !pip install pinecone-client openai pandas openpyxl tqdm

from pinecone import Pinecone, ServerlessSpec
from openai import OpenAI
import pandas as pd
from tqdm import tqdm
import time

# 클라이언트 초기화
pc = Pinecone(api_key=PINECONE_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("[OK] 라이브러리 로드 완료")


[OK] 라이브러리 로드 완료


In [15]:
# ============================================================================
# 3. 데이터 로드 및 전처리 (최적화 버전)
# ============================================================================

df = pd.read_excel(XLSX_PATH, engine='openpyxl')

print(f"전체 row 수: {len(df)}")
print(f"컬럼: {df.columns.tolist()}")

# 메타데이터로 사용할 컬럼들 정의 (acquiror용)
METADATA_COLUMNS = [
    'id', 'update_date', 'acquiror_id', 'acquiror_short_name', 'acquiror_nation',
    'api_flag', 'application_date', 'application_number', 'abstract',
    'invention_name', 'ipc_code'
]

# 필요한 컬럼만 추출 (존재하는 컬럼만)
available_columns = [col for col in METADATA_COLUMNS if col in df.columns]
df_filtered = df[available_columns].copy()

print(f"사용 가능한 컬럼: {available_columns}")

# abstract 기준으로 결측치 제거 (임베딩에 필수)
df_filtered = df_filtered.dropna(subset=['abstract'])
df_filtered = df_filtered[df_filtered['abstract'].str.strip() != '']

# 중복 제거
df_filtered = df_filtered.drop_duplicates()

#
# ---------전체 데이터 사용 (테스트 시 아래 주석 해제)------------------
#TEST_LIMIT = 100
#df_filtered = df_filtered.head(TEST_LIMIT)
# ----------------------------------------------------------------

print(f"유효 데이터 수: {len(df_filtered)}")
df_filtered.head()

전체 row 수: 128471
컬럼: ['id', 'update_date', 'acquiror_id', 'acquiror_short_name', 'acquiror_nation', 'api_flag', 'applicant', 'application_date', 'application_number', 'abstract', 'invention_name', 'ipc_code', 'claim']
사용 가능한 컬럼: ['id', 'update_date', 'acquiror_id', 'acquiror_short_name', 'acquiror_nation', 'api_flag', 'application_date', 'application_number', 'abstract', 'invention_name', 'ipc_code']
유효 데이터 수: 128471


,id,update_date,acquiror_id,acquiror_short_name,acquiror_nation,api_flag,application_date,application_number,abstract,invention_name,ipc_code
0,3164525,2025-11-27 13:59:56.000,28259,Otsuka Pharmaceutical Co Ltd,Japan,KR,20140122,1020187010959,백혈병이나 고형암 등의 암의 진단이나 골수 이식 시기의 결정에 이용할 수 있는 인간...,WT1 mRNA의 발현량 정량 방법,C12Q 1/6886|C12Q 1/6851
1,3164526,2025-11-27 13:59:56.000,28259,Otsuka Pharmaceutical Co Ltd,Japan,KR,20120119,1020137021881,본 발명은 음식품의 변패의 주된 원인균인 내열성 호산성균을 증식시키는 내열성 호산성...,내열성 호산성균 증식 억제 방법,C13B 10/00|C13B 20/16|A23L 1/30
2,3164527,2025-11-27 13:59:56.000,28259,Otsuka Pharmaceutical Co Ltd,Japan,KR,20050912,1020077008268,"본 발명의 목적은 약리 활성 펩티드 및 단백질이 폐점막, 비점막, 구강 점막, 질점...","경점막용 조성물, 및 경점막 흡수의 향상 방법",A61K 38/21|A61K 47/36|A61K 31/7028
3,3164528,2025-11-27 13:59:56.000,28259,Otsuka Pharmaceutical Co Ltd,Japan,KR,20090313,1020107022827,본 발명은 MMP-2 및/또는 MMP-9에 기인하는 질환의 치료에 유용한 안정성이 ...,MMP-2 및/또는 MMP-9 저해제,A61K 31/4439|A61P 11/00
4,3164529,2025-11-27 13:59:56.000,28259,Otsuka Pharmaceutical Co Ltd,Japan,KR,20100226,1020167034405,본 발명은 신규한 만성 동통 치료제를 제공한다. 아리피프라졸을 유효 성분으로서 함유...,만성 동통 치료제,A61K 31/496|A61K 9/00|A61K 47/00


In [ ]:
# ============================================================================
# 4. Pinecone 인덱스 생성 또는 연결
# ============================================================================

# OpenAI text-embedding-3-large: 3072차원
EMBEDDING_DIMENSION = 3072

# 인덱스 존재 여부 확인
existing_indexes = [idx.name for idx in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    print(f"인덱스 '{INDEX_NAME}' 생성 중...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(
            cloud=PINECONE_CLOUD,
            region=PINECONE_REGION
        )
    )
    print(f"[OK] 인덱스 '{INDEX_NAME}' 생성 완료")
else:
    print(f"[OK] 기존 인덱스 '{INDEX_NAME}' 사용")

# 인덱스 연결
index = pc.Index(INDEX_NAME)
print(index.describe_index_stats())


[OK] 기존 인덱스 'acquiror' 사용
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '192',
                                    'content-type': 'application/json',
                                    'date': 'Mon, 26 Jan 2026 12:34:43 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '46',
                                    'x-pinecone-request-id': '8379933535719065756',
                                    'x-pinecone-request-latency-ms': '45',
                                    'x-pinecone-response-duration-ms': '47'}},
 'dimension': 3072,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 999653}},
 'storageFullness': 0.0,
 'total_vector_count': 999653,
 'vector_type': 'dense'}


In [ ]:
# ============================================================================
# 5. 임베딩 생성 함수
# ============================================================================

def get_embedding(text: str, model: str = "text-embedding-3-large") -> list:
    """OpenAI 임베딩 생성"""
    text = text.replace("\n", " ").strip()
    if not text:
        return None
    
    response = openai_client.embeddings.create(
        input=[text],
        model=model
    )
    return response.data[0].embedding


def get_embeddings_batch(texts: list, model: str = "text-embedding-3-large") -> list:
    """배치로 임베딩 생성 (최대 2048개)"""
    # 빈 텍스트 전처리
    processed_texts = [t.replace("\n", " ").strip() if t else "" for t in texts]
    
    response = openai_client.embeddings.create(
        input=processed_texts,
        model=model
    )
    return [item.embedding for item in response.data]


print("[OK] 임베딩 함수 정의 완료")


[OK] 임베딩 함수 정의 완료


In [ ]:
# ============================================================================
# 6. Pinecone에 데이터 업로드 (초고속 병렬 처리 버전)
# ============================================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# 배치 설정 (병렬 처리로 더 큰 단위 사용 가능)
EMBEDDING_BATCH_SIZE = 500   # OpenAI 한 번 호출당 처리량
PARALLEL_WORKERS = 5         # 동시 병렬 처리 수 (OpenAI rate limit 고려)
UPSERT_BATCH_SIZE = 100      # Pinecone 업서트 배치

failed_count = 0
success_count = 0

total_records = len(df_filtered)
print(f"[START] 총 {total_records:,}개 데이터 업로드 시작...")
    
print(f"임베딩 배치: {EMBEDDING_BATCH_SIZE}개씩 / 업서트 배치: {UPSERT_BATCH_SIZE}개씩")
print("=" * 60)

# 시작 시간 기록
from datetime import datetime, timedelta
start_time = datetime.now()

def safe_str(value, max_length=None):
    """안전하게 문자열로 변환"""
    if pd.isna(value):
        return ""
    result = str(value)
    if max_length:
        return result[:max_length]
    return result

# 스레드 안전한 카운터
lock = threading.Lock()

def process_batch(batch_info):
    """단일 배치 처리 함수 (병렬 실행용)"""
    batch_num, batch_start, batch_end = batch_info
    batch_df = df_filtered.iloc[batch_start:batch_end]
    
    try:
        # 배치로 abstract 추출 및 임베딩 생성
        abstracts = batch_df['abstract'].tolist()
        embeddings = get_embeddings_batch(abstracts)
        
        # 벡터 구성
        vectors_to_upsert = []
        for i, (idx, row) in enumerate(batch_df.iterrows()):
            metadata = {
                "id": safe_str(row.get('id', '')),
                "update_date": safe_str(row.get('update_date', '')),
                "acquiror_id": safe_str(row.get('acquiror_id', '')),
                "acquiror_short_name": safe_str(row.get('acquiror_short_name', '')),
                "acquiror_nation": safe_str(row.get('acquiror_nation', '')),
                "api_flag": safe_str(row.get('api_flag', '')),
                "application_date": safe_str(row.get('application_date', '')),
                "application_number": safe_str(row.get('application_number', '')),
                "abstract": safe_str(row.get('abstract', ''), max_length=1000),
                "invention_name": safe_str(row.get('invention_name', ''), max_length=500),
                "ipc_code": safe_str(row.get('ipc_code', ''))
            }
            vectors_to_upsert.append({
                "id": f"{metadata['acquiror_short_name']}_{idx}",
                "values": embeddings[i],
                "metadata": metadata
            })
        
        # Pinecone에 배치 업서트
        for upsert_start in range(0, len(vectors_to_upsert), UPSERT_BATCH_SIZE):
            upsert_end = min(upsert_start + UPSERT_BATCH_SIZE, len(vectors_to_upsert))
            index.upsert(vectors=vectors_to_upsert[upsert_start:upsert_end])
        
        return ("success", len(batch_df), batch_num)
    
    except Exception as e:
        return ("error", len(batch_df), batch_num, str(e))

# 배치 목록 생성
batches = []
batch_num = 1
for batch_start in range(0, total_records, EMBEDDING_BATCH_SIZE):
    batch_end = min(batch_start + EMBEDDING_BATCH_SIZE, total_records)
    batches.append((batch_num, batch_start, batch_end))
    batch_num += 1

total_batches = len(batches)
print(f"[INFO] 총 {total_batches}개 배치를 {PARALLEL_WORKERS}개 병렬 워커로 처리")

# 병렬 처리 실행
completed_batches = 0
with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as executor:
    futures = {executor.submit(process_batch, batch): batch for batch in batches}
    
    for future in as_completed(futures):
        result = future.result()
        
        if result[0] == "success":
            _, count, batch_id = result
            with lock:
                success_count += count
                completed_batches += 1
        else:
            _, count, batch_id, error_msg = result
            with lock:
                failed_count += count
                completed_batches += 1
            print(f"\n[ERROR] batch {batch_id}: {error_msg}")
        
        # 진행 상황 출력 (10배치마다 또는 완료시)
        if completed_batches % 10 == 0 or completed_batches == total_batches:
            with lock:
                elapsed = datetime.now() - start_time
                progress_pct = (success_count / total_records) * 100
                
                if success_count > 0:
                    rate = success_count / elapsed.total_seconds()
                    remaining_secs = (total_records - success_count) / rate
                    remaining_str = str(timedelta(seconds=int(remaining_secs)))
                else:
                    remaining_str = "..."
                
                print(f"[{completed_batches}/{total_batches}] {success_count:,}/{total_records:,}개 "
                      f"({progress_pct:.1f}%) | 경과: {str(elapsed).split('.')[0]} | "
                      f"남은시간: {remaining_str} | 속도: {rate:.1f}/s")

total_elapsed = datetime.now() - start_time
final_rate = success_count / total_elapsed.total_seconds() if total_elapsed.total_seconds() > 0 else 0
print("\n" + "=" * 60)
print(f"[DONE] 업로드 완료!")
print(f"   성공: {success_count:,}개")
print(f"   실패: {failed_count:,}개")
print(f"   소요 시간: {str(total_elapsed).split('.')[0]}")
print(f"   평균 속도: {final_rate:.1f}개/초")
print("=" * 60)

[START] 총 128,471개 데이터 업로드 시작...
임베딩 배치: 500개씩 / 업서트 배치: 100개씩
[INFO] 총 257개 배치를 5개 병렬 워커로 처리
[10/257] 5,000/128,471개 (3.9%) | 경과: 0:00:46 | 남은시간: 0:19:14 | 속도: 107.0/s
[20/257] 10,000/128,471개 (7.8%) | 경과: 0:01:22 | 남은시간: 0:16:16 | 속도: 121.3/s
[30/257] 15,000/128,471개 (11.7%) | 경과: 0:01:52 | 남은시간: 0:14:10 | 속도: 133.4/s
[40/257] 20,000/128,471개 (15.6%) | 경과: 0:02:24 | 남은시간: 0:13:01 | 속도: 138.7/s
[50/257] 25,000/128,471개 (19.5%) | 경과: 0:02:56 | 남은시간: 0:12:09 | 속도: 141.9/s
[60/257] 30,000/128,471개 (23.4%) | 경과: 0:03:33 | 남은시간: 0:11:40 | 속도: 140.5/s
[70/257] 35,000/128,471개 (27.2%) | 경과: 0:04:10 | 남은시간: 0:11:09 | 속도: 139.6/s
[80/257] 40,000/128,471개 (31.1%) | 경과: 0:04:44 | 남은시간: 0:10:29 | 속도: 140.5/s
[90/257] 45,000/128,471개 (35.0%) | 경과: 0:05:20 | 남은시간: 0:09:54 | 속도: 140.5/s
[100/257] 50,000/128,471개 (38.9%) | 경과: 0:06:00 | 남은시간: 0:09:25 | 속도: 138.8/s
[110/257] 55,000/128,471개 (42.8%) | 경과: 0:06:31 | 남은시간: 0:08:43 | 속도: 140.4/s
[120/257] 60,000/128,471개 (46.7%) | 경과: 0:07:02 | 남은시간: 0:08

In [13]:
# ============================================================================
# 7. 업로드 결과 확인
# ============================================================================

stats = index.describe_index_stats()
print(f"인덱스 통계:")
print(f"  - 총 벡터 수: {stats.total_vector_count}")
print(f"  - 차원: {stats.dimension}")


인덱스 통계:
  - 총 벡터 수: 100
  - 차원: 3072


In [ ]:
# ============================================================================
# 8. 검색 테스트 (유사 특허 검색)
# ============================================================================

def search_similar(query_text: str, top_k: int = 5):
    """쿼리 텍스트와 유사한 특허 검색"""
    query_embedding = get_embedding(query_text)
    
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )
    
    print(f"[QUERY] '{query_text[:50]}...'\n")
    for match in results.matches:
        print(f"Score: {match.score:.4f}")
        print(f"  Acquiror: {match.metadata['acquiror_short_name']}")
        print(f"  Abstract: {match.metadata['abstract'][:100]}...")
        print()
    
    return results

# 테스트 검색
# search_similar("로봇 기술 관련 특허")
